### Testar att göra en EDA och olika modeller för vårat dataset

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from src.data import load_raw

df = load_raw()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [2]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [3]:
num = df.copy()
num["TotalCharges"] = pd.to_numeric(num["TotalCharges"], errors="coerce")
num["Churn"] = (num["Churn"] == "Yes").astype(int)
num.corr(numeric_only=True)["Churn"].sort_values(ascending=False)

Churn             1.000000
MonthlyCharges    0.193356
SeniorCitizen     0.150889
TotalCharges     -0.199484
tenure           -0.352229
Name: Churn, dtype: float64

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from src.data import load_dataset
from src.model import train

X, y = load_dataset()

modeller = {
    "Logistic Regression": None,
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "Extra Trees": ExtraTreesClassifier(random_state=42, class_weight="balanced"),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF)": SVC(random_state=42, class_weight="balanced", probability=True),
}

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from src.model import build_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultat = []
for namn, clf in modeller.items():
    score = cross_val_score(build_pipeline(clf), X_train, y_train, cv=cv, scoring="roc_auc")
    resultat.append({"Modell": namn, "ROC-AUC": score.mean(), "Std": score.std()})

tabell = pd.DataFrame(resultat).sort_values("ROC-AUC", ascending=False).round(4)
tabell

/Users/jolle/HT26/Dataview_v2/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/jolle/HT26/Dataview_v2/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/jolle/HT26/Dataview_v2/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/jolle/HT26/Dataview_v2/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The

,Modell,ROC-AUC,Std
5,Gradient Boosting,0.8479,0.0122
4,AdaBoost,0.8472,0.0119
0,Logistic Regression,0.8459,0.0124
6,SVM (RBF),0.8270,0.0090
2,Random Forest,0.8216,0.0124
3,Extra Trees,0.7859,0.0137
1,Decision Tree,0.6604,0.0152


### Sju modeller jämfördes med 5-fold stratifierad korsvalidering på träningsdelen (80 % av datan). Testmängden hölls undan helt under jämförelsen för att undvika att modellvalet optimerades mot den.

### Som utvärderingsmått valdes ROC-AUC framför accuracy. 
### Datasettet är obalanserat med 26,5 % churn, vilket innebär att en modell som alltid predikterar "ingen churn" når 73,5 % accuracy utan att identifiera en enda kund som faktiskt lämnar. 
### ROC-AUC mäter i stället modellens förmåga att rangordna kunder efter risk och påverkas inte av klassfördelningen.

De tre bästa modellerna skiljer sig med 0,002 i ROC-AUC, medan standardavvikelsen mellan folds ligger på omkring 0,012 — sex gånger större. 

Skillnaden är därmed inte urskiljbar från brus. 
Vi behöll logistisk regression, som ger tolkbara koefficienter, tränar på under en sekund och därför kan tränas om vid appstart.

Decision Tree (0,66) presterade märkbart sämre än Random Forest (0,82), trots att båda bygger på samma typ av träd. 
Skillnaden illustrerar hur ensembling motverkar överanpassning.

Jämförelsen gjordes med defaultparametrar. 
Ingen hyperparametersökning genomfördes, vilket innebär att de ensemblebaserade modellerna sannolikt är underskattade.